In [163]:
import plotly.express as px
import pandas as pd
import numpy as np
import pickle
import os

In [168]:
with open('dict_sofipos_sector.pkl', 'rb') as archivo:
    dict_sofipos = pickle.load(archivo)

In [169]:
dict_sofipos = dict_sofipos['fintech']
dict_sofipos.keys()

dict_keys(['Inf_cartera', 'Inf_cartera_E1E2', 'Inf_cartera_E3', 'EPRC', 'Capt_trad', 'Prest_bank', 'CC', 'CG', 'Res_acum', 'Res_ejer', 'Ing_int', 'Ing_int_E1E2', 'Ing_int_E3', 'Ind_financieros', 'Castigos'])

In [170]:
df_cartera = dict_sofipos['Inf_cartera']
df_cartera

,periodo,Cartera de crédito,Créditos comerciales,Créditos consumo,Créditos vivienda,Cartera de crédito_i_b100,Cartera de crédito_pct_YoY,Créditos comerciales_i_b100,Créditos comerciales_pct_YoY,Créditos consumo_i_b100,Créditos consumo_pct_YoY,Créditos vivienda_i_b100,Créditos vivienda_pct_YoY,Créditos comerciales_w,Créditos consumo_w,Créditos vivienda_w,sofipo,fintech
0,2017-01,1565633913.41,587191792.64,969739236.62,8702884.15,66.47,NaN,73.7,NaN,62.82,NaN,58.44,NaN,37.51,61.94,0.56,Fincomún,NO
1,2017-02,1568730959.53,583998852.71,976743262.88,7988843.94,66.6,NaN,73.3,NaN,63.27,NaN,53.65,NaN,37.23,62.26,0.51,Fincomún,NO
2,2017-03,1564748594.05,583828671.87,972909742.9,8010179.28,66.43,NaN,73.27,NaN,63.03,NaN,53.79,NaN,37.31,62.18,0.51,Fincomún,NO
3,2017-04,1562953313.08,587002578.52,968011703.18,7939031.38,66.36,NaN,73.67,NaN,62.71,NaN,53.31,NaN,37.56,61.93,0.51,Fincomún,NO
4,2017-05,1496392715.74,589128989.17,899267924.41,7995802.16,63.53,NaN,73.94,NaN,58.26,NaN,53.69,NaN,39.37,60.1,0.53,Fincomún,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
793,2026-02,48731820790.949997,683836630.89,48047817339.330002,166820.73,361.31,49.89,3171.69,556.31,356.81,48.27,0,52.43,1.4,98.6,0.0,Fintech,SI
794,2026-03,50388826233.280006,812390093.68,49576287775.349998,148364.25,373.6,47.86,3767.93,809.0,368.16,45.86,0,49.27,1.61,98.39,0.0,Fintech,SI
795,2026-04,51914509886.650002,896402987.65,51017981460.199997,125438.8,384.91,47.19,4157.59,1012.04,378.87,44.98,0,-12.59,1.73,98.27,0.0,Fintech,SI
796,2026-05,53533767738.849998,1064681040.99,52468983267.599998,103430.26,396.91,46.66,4938.08,1308.14,389.64,44.04,0,-34.74,1.99,98.01,0.0,Fintech,SI


In [171]:
df_cartera['periodo'] = pd.to_datetime(df_cartera['periodo'], format= '%Y-%m')

In [206]:
def plot_ln(df : pd.DataFrame, 
            serie : str, 
            titulo: str, 
            categorias : str,
            sofipos : list,
            log : bool,
            doble_eje : bool = True,
            eje : str | None = None,
            inicio : str | None = None,
            mostrar : bool = True):

        import pandas as pd
        import numpy as np
        import plotly.express as px
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots

        # Define el inicio de las series
        if pd.isna(inicio) == False:

            df = df[df['periodo'] >= inicio] 

        # Asegura orden cronológico para que el "último valor" sea correcto
        df = df.sort_values('periodo')

        if log == True:
            # Convierte a logaritmo e la serie
            df[serie] = df[serie].astype(float)
            df[f'{serie}_ln'] = np.log(df[serie])
        
            # Selecciona las sofipos 
            df = df[df['sofipo'].isin(sofipos)]

            y_col = f'{serie}_ln'

        else:
            # Selecciona las sofipos 
            df = df[df['sofipo'].isin(sofipos)]
            y_col = serie
        
        fig_px = px.line(
            df, 
            x="periodo", 
            y=y_col, 
            title= titulo,
            color=categorias)

        if doble_eje:
            # Construye la figura con doble eje
            fig = make_subplots(specs=[[{"secondary_y": True}]])

            for trace in fig_px.data:
                es_total = trace.name in ['Total SOFIPOS', 'Fintech']
                fig.add_trace(trace, secondary_y=not es_total)

            fig.update_layout(
                title=titulo,
                xaxis=dict(
                    title=None,
                    tickformat="%Y",
                    dtick="M12",
                    hoverformat="%Y-%m")
                )
            
            fig.update_yaxes(title_text="Sector SOFIPOS", secondary_y=False)
            fig.update_yaxes(title_text="Otras SOFIPOS", secondary_y=True)

            # Líneas de referencia en y = 100 para cada eje
            fig.add_hline(y=100, line_dash="dash", line_color="black", secondary_y=False)
            fig.add_hline(y=100, line_dash="dash", line_color="darkblue", secondary_y=True)

        else:
            # Figura de un solo eje
            fig = fig_px

            fig.update_layout(
                title=titulo,
                xaxis=dict(
                    title=None,
                    tickformat="%Y",
                    dtick="M12",
                    hoverformat="%Y-%m")
                )

            fig.update_yaxes(
                side="right",
                showticklabels=True,
                title_text= eje
            )

            # Línea de referencia en y = 100
            fig.add_hline(y=100, line_dash="dash", line_color="black")

        # Leyenda dentro del gráfico, esquina superior izquierda
        fig.update_layout(
            legend=dict(
                title=dict(text="SOFIPO"),
                x=0.01,
                y=0.99,
                xanchor="left",
                yanchor="top",
                bgcolor="rgba(255,255,255,0.6)"
            )
        )

        # Agrega el último valor y fecha a la leyenda de cada serie
        def actualizar_leyenda(trace):
            x_vals = trace.x
            y_vals = trace.y

            if len(x_vals) == 0:
                return

            ultimo_x = pd.to_datetime(x_vals[-1]).strftime('%Y-%m')
            ultimo_y = y_vals[-1]

            trace.update(name=f"{trace.name} | {ultimo_y:.2f} ({ultimo_x})")

        fig.for_each_trace(actualizar_leyenda)

        if mostrar:
            fig.show()

        return fig

In [ ]:
df_cartera['sofipo'].unique()

In [199]:
plt_sector = plot_ln(df = df_cartera,
                serie= 'Cartera de crédito_i_b100',
                titulo= '',
                categorias= 'sofipo',
                sofipos= ['Total SOFIPOS', 'Fintech'],
                doble_eje = False,
                log=False,
                eje = 'Cartera Total de Crédito (2023-01 = 100)',
                inicio= '2022-01')

In [198]:
plt_sofipo = plot_ln(df = df_cartera,
                serie= 'Cartera de crédito_i_b100',
                titulo= '',
                categorias= 'sofipo',
                sofipos= ['Fincomún', 'Tamazula', 'Libertad', 'Crediclub'],
                doble_eje = False,
                log=False,
                eje = 'Cartera Total de Crédito (2023-01 = 100)',
                inicio= '2022-01')

In [204]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

panel = make_subplots(rows=2, cols=1, subplot_titles=("Sector", 
                                                      "SOFIPO"))

plt_sector = plot_ln(df = df_cartera,
                serie= 'Cartera de crédito_i_b100',
                titulo= '',
                categorias= 'sofipo',
                sofipos= ['Total SOFIPOS', 'Fintech'],
                doble_eje = False,
                log=False,
                eje = 'Cartera Total de Crédito (2023-01 = 100)',
                inicio= '2022-01')

plt_sofipo = plot_ln(df = df_cartera,
                serie= 'Cartera de crédito_i_b100',
                titulo= '',
                categorias= 'sofipo',
                sofipos= ['Fincomún', 'Tamazula', 'Libertad', 'Crediclub'],
                doble_eje = False,
                log=False,
                eje = 'Cartera Total de Crédito (2023-01 = 100)',
                inicio= '2022-01')

for trace in plt_sector.data:
    panel.add_trace(trace, row=1, col=1)

for trace in plt_sofipo.data:
    panel.add_trace(trace, row=2, col=1)

panel.update_layout(title="Cartera Total de Crédito", 
                    showlegend=True,
                    width=1200,
                    height=900)
panel.show()

In [212]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

panel = make_subplots(rows=2, cols=1, subplot_titles=("Sector", 
                                                      "SOFIPO"))

plt_sector = plot_ln(df = df_cartera,
                serie= 'Cartera de crédito_i_b100',
                titulo= '',
                categorias= 'sofipo',
                sofipos= ['Total SOFIPOS', 'Fintech'],
                doble_eje = False,
                log=False,
                eje = 'Cartera Total de Crédito (2023-01 = 100)',
                inicio= '2022-01',
                mostrar=False)

plt_sofipo = plot_ln(df = df_cartera,
                serie= 'Cartera de crédito_i_b100',
                titulo= '',
                categorias= 'sofipo',
                sofipos= ['Fincomún', 'Tamazula', 'Libertad', 'Crediclub'],
                doble_eje = False,
                log=False,
                eje = 'Cartera Total de Crédito (2023-01 = 100)',
                inicio= '2022-01',
                mostrar=False)

# Traces del primer gráfico → leyenda 1
for trace in plt_sector.data:
    trace.legend = "legend"
    panel.add_trace(trace, row=1, col=1)

# Traces del segundo gráfico → leyenda 2
for trace in plt_sofipo.data:
    trace.legend = "legend2"
    panel.add_trace(trace, row=2, col=1)

# Replica el formato del eje X en ambas filas
panel.update_xaxes(title=None, tickformat="%Y", dtick="M12", hoverformat="%Y-%m",
                    row=1, col=1)
panel.update_xaxes(title=None, tickformat="%Y", dtick="M12", hoverformat="%Y-%m",
                    row=2, col=1)

# Replica el título y posición del eje Y en ambas filas
panel.update_yaxes(side="right", showticklabels=True,
                    title_text='Cartera Total de Crédito (2023-01 = 100)',
                    row=1, col=1)
panel.update_yaxes(side="right", showticklabels=True,
                    title_text='Cartera Total de Crédito (2023-01 = 100)',
                    row=2, col=1)

# Replica las líneas de referencia en y=100 en ambas filas
panel.add_hline(y=100, line_dash="dash", line_color="black", row=1, col=1)
panel.add_hline(y=100, line_dash="dash", line_color="black", row=2, col=1)

# Layout general del panel
panel.update_layout(
    title="Cartera Total de Crédito",
    width=1200,
    height=900,
    legend=dict(
        title=dict(text="SOFIPO"),
        x=0.01, y=0.99,
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.6)"
    ),
    legend2=dict(
        title=dict(text="SOFIPO"),
        x=0.01, y=0.45,   # ajusta 'y' para que caiga junto al segundo subplot
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.6)"
    )
)
panel.show()

In [ ]:
df_cartera['periodo'] = pd.to_datetime(df_cartera['periodo'].astype(str), format='%Y-%m-%d').dt.to_period('M')

In [ ]:
df_cartera[df_cartera['periodo'] >= '2022-02']